# Notebook 11 — Window Drift: Rolling Prime-Lane Trajectories

**Repo:** `mod30-residue-lanes`  
**Layer:** `notebooks/`

Notebook 01 established the symmetric admissible-lane baseline.

Notebook 07 introduced prime-filtered asymmetry.

Notebook 11 introduces window drift: rolling windows move across the prime-filtered lane substrate, making lane trajectories visible.

Constraint view:

> Window drift turns static lane counts into rolling eight-lane trajectories, revealing how prime-filtered structure changes under shifted observation boundaries.


## Goals

1. Detect repo root and create standard output directories.
2. Load MRL foundations from Notebook 00 when available.
3. Generate prime-filtered admissible-lane data.
4. Build rolling windows with step size smaller than window size.
5. Track lane `11` as the focal drift lane.
6. Measure rolling vectors, drift magnitude, similarity, and lane leadership.
7. Export CSV, JSON, figures, and Markdown report.
8. Create a Colab-downloadable output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "src" / "mod30_residue_lanes").exists() or (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [NOTEBOOKS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Load Notebook 00 foundation output

In [ ]:
foundation_path = RESULTS_DIR / "notebook00_mrl_foundations.json"

if foundation_path.exists():
    foundation = json.loads(foundation_path.read_text())
    MODULUS = foundation["modulus"]
    ADMISSIBLE_RESIDUES = foundation["admissible_residues"]
    print("Loaded foundation:", foundation_path)
else:
    MODULUS = 30
    ADMISSIBLE_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
    print("Notebook 00 foundation not found; using defaults.")

TARGET_LANE = 11
LANE_LABEL = f"{TARGET_LANE:02d}"

print("MODULUS:", MODULUS)
print("ADMISSIBLE_RESIDUES:", ADMISSIBLE_RESIDUES)
print("TARGET_LANE:", LANE_LABEL)


## Prime helper

In [ ]:
def prime_sieve(n_max: int) -> np.ndarray:
    if n_max < 2:
        return np.zeros(n_max + 1, dtype=bool)

    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    limit = int(np.sqrt(n_max)) + 1
    for p in range(2, limit):
        if sieve[p]:
            sieve[p*p:n_max+1:p] = False

    return sieve


def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


## Generate prime-filtered lane data

Notebook 11 uses a larger interval and rolling windows.

The key change from Notebook 07:

```text
fixed non-overlapping windows → rolling overlapping windows
```


In [ ]:
N_MAX = 60000
WINDOW_SIZE = 3000
STEP_SIZE = 300

values = np.arange(1, N_MAX + 1)
is_prime_array = prime_sieve(N_MAX)

df = pd.DataFrame({
    "n": values,
    "residue": values % MODULUS,
})

df["is_admissible"] = df["residue"].isin(ADMISSIBLE_RESIDUES)
df["is_prime"] = is_prime_array[values]
df["is_admissible_prime"] = df["is_admissible"] & df["is_prime"]
df["is_target_lane"] = df["residue"] == TARGET_LANE
df["cycle"] = (df["n"] - 1) // MODULUS

admissible_prime_df = df[df["is_admissible_prime"]].copy()
lane11_prime_df = df[df["is_admissible_prime"] & df["is_target_lane"]].copy()

print("Total values:", len(df))
print("Admissible prime values:", len(admissible_prime_df))
print("Lane 11 prime values:", len(lane11_prime_df))
print("Window size:", WINDOW_SIZE)
print("Step size:", STEP_SIZE)


## Rolling window lane vectors

Each rolling window becomes an eight-entry vector ordered by:

```text
01, 07, 11, 13, 17, 19, 23, 29
```


In [ ]:
lane_cols = [f"lane_{r:02d}" for r in ADMISSIBLE_RESIDUES]
window_rows = []

for start in range(1, N_MAX - WINDOW_SIZE + 2, STEP_SIZE):
    stop = start + WINDOW_SIZE - 1
    window = df[(df["n"] >= start) & (df["n"] <= stop)]
    prime_window = window[window["is_admissible_prime"]]
    counts = prime_window.groupby("residue").size().reindex(ADMISSIBLE_RESIDUES, fill_value=0)

    row = {
        "window_start": start,
        "window_stop": stop,
        "window_center": (start + stop) / 2,
        "window_size": len(window),
        "prime_count": int(prime_window.shape[0]),
    }
    for residue, count in counts.items():
        row[f"lane_{residue:02d}"] = int(count)

    total = max(1, int(counts.sum()))
    for residue, count in counts.items():
        row[f"share_{residue:02d}"] = float(count / total)

    row["target_lane_count"] = int(counts.loc[TARGET_LANE])
    row["target_lane_share"] = float(counts.loc[TARGET_LANE] / total)
    row["leader_lane"] = f"{int(counts.idxmax()):02d}"
    row["leader_count"] = int(counts.max())
    row["leader_margin"] = int(counts.max() - counts.sort_values(ascending=False).iloc[1])
    window_rows.append(row)

rolling_vectors = pd.DataFrame(window_rows)

rolling_csv = RESULTS_DIR / "notebook11_rolling_prime_lane_vectors.csv"
rolling_vectors.to_csv(rolling_csv, index=False)

print(rolling_csv)
rolling_vectors.head()


## Drift metrics

Drift is measured between consecutive rolling lane-count vectors.

- Euclidean drift: magnitude of count-vector change
- Cosine drift: `1 - cosine_similarity(previous, current)`


In [ ]:
matrix = rolling_vectors[lane_cols].to_numpy(dtype=float)

drift_rows = []
for i in range(1, len(matrix)):
    prev_vec = matrix[i - 1]
    cur_vec = matrix[i]

    drift_rows.append({
        "window_start": int(rolling_vectors.loc[i, "window_start"]),
        "previous_window_start": int(rolling_vectors.loc[i - 1, "window_start"]),
        "euclidean_drift": float(np.linalg.norm(cur_vec - prev_vec)),
        "cosine_similarity_previous": cosine_similarity(prev_vec, cur_vec),
        "cosine_drift": float(1 - cosine_similarity(prev_vec, cur_vec)),
        "target_lane_count": int(rolling_vectors.loc[i, "target_lane_count"]),
        "target_lane_share": float(rolling_vectors.loc[i, "target_lane_share"]),
        "leader_lane": rolling_vectors.loc[i, "leader_lane"],
    })

drift_df = pd.DataFrame(drift_rows)

drift_csv = RESULTS_DIR / "notebook11_rolling_drift_metrics.csv"
drift_df.to_csv(drift_csv, index=False)

print(drift_csv)
drift_df.head()


## Lane similarity under rolling drift

Similarity is computed from each lane's rolling count trajectory.


In [ ]:
target_trajectory = rolling_vectors[f"lane_{TARGET_LANE:02d}"].to_numpy()

similarity_rows = []
for r in ADMISSIBLE_RESIDUES:
    col = f"lane_{r:02d}"
    sim = cosine_similarity(target_trajectory, rolling_vectors[col].to_numpy())
    corr = float(np.corrcoef(target_trajectory, rolling_vectors[col].to_numpy())[0, 1])
    similarity_rows.append({
        "target_lane": LANE_LABEL,
        "comparison_lane": f"{r:02d}",
        "cosine_similarity": sim,
        "pearson_correlation": corr,
    })

similarity_df = pd.DataFrame(similarity_rows)

similarity_csv = RESULTS_DIR / "notebook11_lane11_rolling_similarity.csv"
similarity_df.to_csv(similarity_csv, index=False)

print(similarity_csv)
similarity_df


## Lane leadership summary

A leader lane is the lane with the highest prime count in a rolling window.


In [ ]:
leadership_df = (
    rolling_vectors
    .groupby("leader_lane")
    .agg(
        leadership_windows=("leader_lane", "size"),
        mean_leader_margin=("leader_margin", "mean"),
    )
    .reset_index()
    .sort_values("leader_lane")
)

leadership_csv = RESULTS_DIR / "notebook11_lane_leadership_summary.csv"
leadership_df.to_csv(leadership_csv, index=False)

print(leadership_csv)
leadership_df


## Summary exports

In [ ]:
summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "11_lane_residue_11",
    "title": "Window Drift: Rolling Prime-Lane Trajectories",
    "modulus": MODULUS,
    "target_lane": TARGET_LANE,
    "target_lane_label": LANE_LABEL,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "n_max": N_MAX,
    "window_size": WINDOW_SIZE,
    "step_size": STEP_SIZE,
    "rolling_windows": int(len(rolling_vectors)),
    "admissible_prime_values": int(len(admissible_prime_df)),
    "target_lane_prime_values": int(len(lane11_prime_df)),
    "mean_target_lane_count": float(rolling_vectors["target_lane_count"].mean()),
    "std_target_lane_count": float(rolling_vectors["target_lane_count"].std(ddof=0)),
    "mean_euclidean_drift": float(drift_df["euclidean_drift"].mean()),
    "max_euclidean_drift": float(drift_df["euclidean_drift"].max()),
    "mean_cosine_drift": float(drift_df["cosine_drift"].mean()),
    "max_cosine_drift": float(drift_df["cosine_drift"].max()),
    "constraint_view": "Window drift turns static lane counts into rolling eight-lane trajectories, revealing how prime-filtered structure changes under shifted observation boundaries.",
}

json_path = RESULTS_DIR / "notebook11_window_drift_summary.json"
json_path.write_text(json.dumps(summary, indent=2))

print(json_path)
print(json.dumps(summary, indent=2))


## Figures

In [ ]:
figure_names = []

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_names.append(name)
    print(path)

# 1. Rolling prime lane heatmap
matrix = rolling_vectors[lane_cols].to_numpy()
plt.figure(figsize=(11, 6))
plt.imshow(matrix, aspect="auto")
plt.xticks(range(len(lane_cols)), [c.replace("lane_", "") for c in lane_cols])
plt.yticks(
    range(0, len(rolling_vectors), max(1, len(rolling_vectors)//10)),
    rolling_vectors["window_start"].iloc[::max(1, len(rolling_vectors)//10)]
)
plt.xlabel("Residue lane")
plt.ylabel("Rolling window start")
plt.colorbar(label="Prime count")
plt.title("Notebook 11 — Rolling Prime Lane Vectors")
savefig("notebook11_rolling_prime_lane_vectors.png")

# 2. Lane 11 rolling count trajectory
plt.figure(figsize=(12, 5))
plt.plot(rolling_vectors["window_center"], rolling_vectors[f"lane_{TARGET_LANE:02d}"])
plt.xlabel("Window center")
plt.ylabel("Lane 11 prime count")
plt.title("Notebook 11 — Lane 11 Rolling Prime Count")
savefig("notebook11_lane11_rolling_count.png")

# 3. All lane trajectories
plt.figure(figsize=(12, 5))
for r in ADMISSIBLE_RESIDUES:
    plt.plot(rolling_vectors["window_center"], rolling_vectors[f"lane_{r:02d}"], label=f"{r:02d}")
plt.xlabel("Window center")
plt.ylabel("Prime count")
plt.title("Notebook 11 — Rolling Prime Count Trajectories")
plt.legend(ncol=4)
savefig("notebook11_all_lane_rolling_trajectories.png")

# 4. Euclidean drift timeline
plt.figure(figsize=(12, 5))
plt.plot(drift_df["window_start"], drift_df["euclidean_drift"])
plt.xlabel("Rolling window start")
plt.ylabel("Euclidean drift")
plt.title("Notebook 11 — Rolling Vector Drift Magnitude")
savefig("notebook11_euclidean_drift_timeline.png")

# 5. Cosine drift timeline
plt.figure(figsize=(12, 5))
plt.plot(drift_df["window_start"], drift_df["cosine_drift"])
plt.xlabel("Rolling window start")
plt.ylabel("Cosine drift")
plt.title("Notebook 11 — Rolling Vector Cosine Drift")
savefig("notebook11_cosine_drift_timeline.png")

# 6. Lane 11 similarity
plt.figure(figsize=(8, 4))
plt.bar(similarity_df["comparison_lane"], similarity_df["pearson_correlation"])
plt.xlabel("Comparison lane")
plt.ylabel("Pearson correlation with lane 11")
plt.title("Notebook 11 — Lane 11 Rolling Correlation")
savefig("notebook11_lane11_rolling_correlation.png")

# 7. Leadership windows
plt.figure(figsize=(8, 4))
plt.bar(leadership_df["leader_lane"], leadership_df["leadership_windows"])
plt.xlabel("Leader lane")
plt.ylabel("Leadership windows")
plt.title("Notebook 11 — Lane Leadership Across Rolling Windows")
savefig("notebook11_lane_leadership_windows.png")


## Build Markdown report

The report uses repo-relative links for CSV, JSON, and figure outputs.


In [ ]:
report_path = REPORTS_DIR / "report_11_window_drift.md"

output_links = "\n".join([
    '- Rolling prime lane vectors CSV: <a href="../results/notebook11_rolling_prime_lane_vectors.csv">`results/notebook11_rolling_prime_lane_vectors.csv`</a>',
    '- Rolling drift metrics CSV: <a href="../results/notebook11_rolling_drift_metrics.csv">`results/notebook11_rolling_drift_metrics.csv`</a>',
    '- Lane 11 rolling similarity CSV: <a href="../results/notebook11_lane11_rolling_similarity.csv">`results/notebook11_lane11_rolling_similarity.csv`</a>',
    '- Lane leadership summary CSV: <a href="../results/notebook11_lane_leadership_summary.csv">`results/notebook11_lane_leadership_summary.csv`</a>',
    '- Summary JSON: <a href="../results/notebook11_window_drift_summary.json">`results/notebook11_window_drift_summary.json`</a>',
] + [
    f'- Figure: <a href="../figures/{name}">`figures/{name}`</a>'
    for name in figure_names
])

report = f"""# Report 11 — Window Drift: Rolling Prime-Lane Trajectories

Notebook 11 introduces rolling window drift after Notebook 01 established symmetry and Notebook 07 introduced prime-filtered asymmetry.

Constraint view:

> Window drift turns static lane counts into rolling eight-lane trajectories, revealing how prime-filtered structure changes under shifted observation boundaries.

## Generated outputs

{output_links}

## Summary

| Metric | Value |
|---|---:|
| Modulus | {MODULUS} |
| Target lane | {LANE_LABEL} |
| Interval max | {N_MAX} |
| Window size | {WINDOW_SIZE} |
| Step size | {STEP_SIZE} |
| Rolling windows | {len(rolling_vectors)} |
| Admissible prime values | {len(admissible_prime_df)} |
| Lane 11 prime values | {len(lane11_prime_df)} |
| Mean lane 11 rolling count | {rolling_vectors["target_lane_count"].mean():.6f} |
| Std lane 11 rolling count | {rolling_vectors["target_lane_count"].std(ddof=0):.6f} |
| Mean Euclidean drift | {drift_df["euclidean_drift"].mean():.6f} |
| Max Euclidean drift | {drift_df["euclidean_drift"].max():.6f} |
| Mean cosine drift | {drift_df["cosine_drift"].mean():.8f} |
| Max cosine drift | {drift_df["cosine_drift"].max():.8f} |

## Lane 11 rolling similarity

{similarity_df.to_markdown(index=False)}

## Lane leadership summary

{leadership_df.to_markdown(index=False)}

## Drift summary

{drift_df[["euclidean_drift", "cosine_similarity_previous", "cosine_drift", "target_lane_count", "target_lane_share"]].describe().to_markdown()}

## Interpretation

- Notebook 01 showed equal lane counts under complete modulo-30 cycles.
- Notebook 07 showed prime-filtered asymmetry under fixed windows.
- Notebook 11 turns prime-filtered counts into rolling trajectories.
- Drift metrics measure how much the eight-lane vector changes as observation boundaries move.
- Lane leadership summarizes which residue lane locally dominates a rolling prime-count window.
- Lane `11` becomes the first window-drift focal lane.

## Next step

Notebook 13 can study boundary effects: how local windows behave near pre-15 constraint boundary lane `13`.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:5000])


## Render generated figures in notebook

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook11_window_drift_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook11_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in FIGURES_DIR.glob("notebook11_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in REPORTS_DIR.glob("report_11_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook11_window_drift_outputs.zip"
# export_path = REPO_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook11_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#         for p in folder.glob("report_11_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
